# Lab Iceberg Format Versions — V1 vs V2 vs V3

**Profil docker :** `iceberg` · **Durée :** 45 min · **Position dans le programme :** J1 — après Iceberg internals

---

## Objectifs
1. Comprendre les différences structurelles entre V1, V2 et V3 dans les metadata files
2. Mesurer l'impact de chaque version sur les opérations DML (DELETE, UPDATE, MERGE)
3. Observer les Deletion Vectors V3 face aux positional delete files V2
4. Tester le type `Variant` (V3) et ses cas d'usage ML
5. Maîtriser la migration V1→V2→V3 et ses points de non-retour

---

## Règle d'or

| Version | Quand l'utiliser |
|---------|------------------|
| **V1** | Tables append-only (logs, events). Compatibilité maximale engines. |
| **V2** | **Standard de production.** Dès qu'il y a DELETE, UPDATE, MERGE ou RGPD. |
| **V3** | ML features semi-structurées (Variant), haute fréquence de deletes (Deletion Vectors). Uniquement si Spark/PyIceberg exclusif. |

> **⚠️ Point critique :** La migration V2 → V3 est **irréversible**.
> Trino et Flink (2024) ne lisent pas les tables V3.
> Vérifier la matrice de compatibilité engines **avant** toute migration en production.

In [1]:
import os, json, boto3, time, requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ── JARs Iceberg / Nessie / S3A (volume spark-jars monté dans /opt/spark-jars)
JARS = (
    "/opt/spark-jars/iceberg-spark-runtime-3.5_2.12-1.11.0.jar,"
    "/opt/spark-jars/iceberg-aws-bundle-1.11.0.jar,"
    "/opt/spark-jars/nessie-spark-extensions-3.5_2.12-0.105.3.jar,"
    "/opt/spark-jars/hadoop-aws-3.3.4.jar,"
    "/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
)

# ── SparkSession ──────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("Iceberg-Format-Versions")
    .config("spark.jars", JARS)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")
    # Catalog Polaris (REST)
    .config("spark.sql.catalog.polaris",              "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.polaris.catalog-impl", "org.apache.iceberg.rest.RESTCatalog")
    .config("spark.sql.catalog.polaris.uri",          os.getenv("POLARIS_URI", "http://polaris:8181/api/catalog"))
    .config("spark.sql.catalog.polaris.warehouse",    "retailco")         # nom du catalog Polaris
    .config("spark.sql.catalog.polaris.credential",   os.getenv("POLARIS_CREDENTIAL", "root:s3cr3t"))
    .config("spark.sql.catalog.polaris.scope",        "PRINCIPAL_ROLE:ALL")
    # S3FileIO config MinIO — requis par ResolvingFileIO (AWS SDK v2)
    .config("spark.sql.catalog.polaris.s3.endpoint",          os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.sql.catalog.polaris.s3.access-key-id",     os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.sql.catalog.polaris.s3.secret-access-key", os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.sql.catalog.polaris.s3.path-style-access",  "true")
    .config("spark.sql.catalog.nessie.s3.endpoint",           os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.sql.catalog.nessie.s3.access-key-id",      os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.sql.catalog.nessie.s3.secret-access-key",  os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.sql.catalog.nessie.s3.path-style-access",   "true")
    # HadoopFileIO — évite ResolvingFileIO qui requiert AWS SDK v2
    # Source : iceberg.apache.org/docs/latest/aws/#s3-file-io
    # Catalog Nessie
    .config("spark.sql.catalog.nessie",              "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri",          os.getenv("NESSIE_URI_V1", "http://nessie:19120/api/v1"))
    .config("spark.sql.catalog.nessie.ref",          "main")
    .config("spark.sql.catalog.nessie.warehouse",    "s3a://warehouse/nessie/")
    # MinIO / S3A
    .config("spark.hadoop.fs.s3a.endpoint",             os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.hadoop.fs.s3a.access.key",           os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.hadoop.fs.s3a.secret.key",           os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.hadoop.fs.s3a.path.style.access",    "true")
    .config("spark.hadoop.fs.s3a.impl",                 "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.defaultCatalog", "polaris")
    .getOrCreate()
)

print(f"✅ Spark {spark.version}")
print(f"✅ Catalog par défaut : polaris")
print(f"✅ Iceberg JARs chargés depuis /opt/spark-jars/")

# ── Client S3 pour inspecter les fichiers MinIO ───────────────────────────────
MINIO = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
s3 = boto3.client("s3", endpoint_url=MINIO,
    aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")

def list_metadata_files(namespace, table, bucket="warehouse"):
    """Lister les fichiers générés par Iceberg dans MinIO."""
    prefix = f"{namespace}/{table}/metadata/"
    try:
        resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
        files = resp.get("Contents", [])
        by_type = {}
        for f in files:
            ext = f["Key"].split(".")[-1]
            by_type.setdefault(ext, []).append(f["Key"].split("/")[-1])
        return by_type
    except Exception as e:
        return {"error": str(e)}
print("✅ Helpers MinIO prêts")

def count_delete_files(namespace, table):
    """Retourne un DataFrame Spark groupé par type de fichier (content) pour une table Iceberg.
    content : 0 = data, 1 = positional deletes, 2 = equality deletes.
    Colonnes retournées : content, n (count), total_bytes (somme des tailles)."""
    return spark.sql(f"""
        SELECT content, count(*) AS n, sum(file_size_in_bytes) AS total_bytes
        FROM polaris.{namespace}.{table}.files
        GROUP BY content
    """)
print("✅ count_delete_files() prête")

✅ Spark 3.5.0
✅ Catalog par défaut : polaris
✅ Iceberg JARs chargés depuis /opt/spark-jars/
✅ Helpers MinIO prêts
✅ count_delete_files() prête


---
## ⏱️ Temps 1 — Kata d'amorce (10 min)

Créer les 3 tables avec des données identiques, observer les propriétés de format dans les metadata.

In [2]:
spark.sql("USE polaris")
spark.sql("CREATE NAMESPACE IF NOT EXISTS versions_demo")

# ── Données RetailCo communes aux 3 versions ──────────────────────────────────
# Schéma identique — seul le format-version change

COMMON_DDL = """
    transaction_id  STRING,
    customer_id     STRING,
    store_country    STRING,
    category        STRING,
    unit_price_xof  BIGINT,
    quantity        INT,
    total_amount_xof BIGINT,
    return_flag     BOOLEAN,
    transaction_ts  TIMESTAMP
"""

COMMON_DATA = """
    VALUES
    ('txn-001','cust-001','Cote d Ivoire','Electronique',395000,1,466100,false,current_timestamp()),
    ('txn-002','cust-002','Senegal','Vetements',32000,2,75520,false,current_timestamp()),
    ('txn-003','cust-003','Togo','Maison',18000,3,63720,false,current_timestamp()),
    ('txn-004','cust-001','Cote d Ivoire','Sport',52000,1,134424,false,current_timestamp()),
    ('txn-005','cust-004','Grand Est','Electronique',84700,1,99839,false,current_timestamp())
"""

for version in [1, 2, 3]:
    table = f'transactions_v{version}'
    # Drop si existe
    spark.sql(f'DROP TABLE IF EXISTS polaris.versions_demo.{table}')
    # Créer avec le bon format-version
    spark.sql(f"""
        CREATE TABLE polaris.versions_demo.{table} ({COMMON_DDL})
        USING iceberg
        TBLPROPERTIES (
            'format-version' = '{version}',
            'write.format.default' = 'parquet'
        )
    """)
    # Insérer les données
    spark.sql(f'INSERT INTO polaris.versions_demo.{table} {COMMON_DATA}')
    print(f'✅ Table V{version} créée : polaris.versions_demo.{table}')

print('\n📊 Vérification des format-versions :')
for version in [1, 2, 3]:
    table = f'transactions_v{version}'
    try:
        # SHOW TBLPROPERTIES est supporté par Spark+Iceberg pour toutes les versions
        props = spark.sql(f"SHOW TBLPROPERTIES polaris.versions_demo.{table}").collect()
        fv = next((r['value'] for r in props if r['key'] == 'format-version'), 'inconnu')
        print(f'   transactions_v{version} : format-version = {fv}')
    except Exception as e:
        print(f'   V{version}: {e}')

✅ Table V1 créée : polaris.versions_demo.transactions_v1
✅ Table V2 créée : polaris.versions_demo.transactions_v2
✅ Table V3 créée : polaris.versions_demo.transactions_v3

📊 Vérification des format-versions :
   transactions_v1 : format-version = 1
   transactions_v2 : format-version = 2
   transactions_v3 : format-version = 3


In [3]:
# ── Observer les differences dans les metadata files ─────────────────────────
print('📁 Structure des metadata files par version :\n')
for version in [1, 2, 3]:
    table = f'transactions_v{version}'
    print(f'  V{version} — polaris.versions_demo.{table}/metadata/ :')
    list_metadata_files('versions_demo', table)
    # Lire le dernier metadata.json pour voir le format-version
    try:
        prefix = f'versions_demo/{table}/metadata/'
        resp = s3.list_objects_v2(Bucket='warehouse', Prefix=prefix)
        jsons = sorted([o['Key'] for o in resp.get('Contents', []) if o['Key'].endswith('.json')])
        if jsons:
            meta = json.loads(s3.get_object(Bucket='warehouse', Key=jsons[-1])['Body'].read())
            print(f'     format-version  : {meta.get("format-version")}')
            print(f'     schemas         : {len(meta.get("schemas", []))}')
            print(f'     snapshots       : {len(meta.get("snapshots", []))}')
    except Exception as e:
        print(f'     (metadata non accessible: {e})')
    print()

📁 Structure des metadata files par version :

  V1 — polaris.versions_demo.transactions_v1/metadata/ :
     format-version  : 1
     schemas         : 1
     snapshots       : 2

  V2 — polaris.versions_demo.transactions_v2/metadata/ :
     format-version  : 2
     schemas         : 1
     snapshots       : 2

  V3 — polaris.versions_demo.transactions_v3/metadata/ :
     format-version  : 3
     schemas         : 1
     snapshots       : 2



---
## ⏱️ Temps 2 — Incident à résoudre (25 min)

### Partie A — DELETE sur V1, V2, V3 : observer les différences (15 min)

> **Contexte RGPD** : RetailCo doit supprimer toutes les transactions du client `cust-001`.
> Comparer ce qui se passe réellement au niveau des fichiers pour chaque version.

In [4]:
# ── DELETE sur V1 ─────────────────────────────────────────────────────────────
# En V1, il n'existe pas de row-level delete.
# Spark va simuler le DELETE en réécrivant tout le fichier de données (CoW implicite).

TARGET = 'cust-001'

print('=' * 55)
print('DELETE V1 — transactions_v1')
print('=' * 55)

# Fichiers avant
files_before_v1 = spark.sql('SELECT count(*) AS n FROM polaris.versions_demo.transactions_v1.files').collect()[0]['n']
snapshots_before_v1 = spark.sql('SELECT count(*) AS n FROM polaris.versions_demo.transactions_v1.snapshots').collect()[0]['n']

t0 = time.time()
spark.sql(f"DELETE FROM polaris.versions_demo.transactions_v1 WHERE customer_id = '{TARGET}'")
elapsed_v1 = round(time.time() - t0, 3)

files_after_v1  = spark.sql('SELECT count(*) AS n FROM polaris.versions_demo.transactions_v1.files').collect()[0]['n']
snapshots_after_v1 = spark.sql('SELECT count(*) AS n FROM polaris.versions_demo.transactions_v1.snapshots').collect()[0]['n']
remaining_v1 = spark.sql(f"SELECT count(*) AS n FROM polaris.versions_demo.transactions_v1 WHERE customer_id = '{TARGET}'").collect()[0]['n']

print(f'  Temps d\'exécution   : {elapsed_v1}s')
print(f'  Fichiers data       : {files_before_v1} → {files_after_v1} (réécriture complète)')
print(f'  Snapshots           : {snapshots_before_v1} → {snapshots_after_v1}')
print(f'  Lignes restantes    : {remaining_v1} (attendu: 0)')
print(f'  Mécanisme           : Copy-on-Write implicite — tout le fichier réécrit')
print(f'  ⚠️  En V1, le DELETE réécrit TOUJOURS tous les fichiers affectés.')

DELETE V1 — transactions_v1
  Temps d'exécution   : 11.06s
  Fichiers data       : 4 → 3 (réécriture complète)
  Snapshots           : 1 → 2
  Lignes restantes    : 0 (attendu: 0)
  Mécanisme           : Copy-on-Write implicite — tout le fichier réécrit
  ⚠️  En V1, le DELETE réécrit TOUJOURS tous les fichiers affectés.


In [5]:
# ── DELETE sur V2 (MoR) ───────────────────────────────────────────────────────
# En V2 avec MoR, le DELETE écrit un positional delete file
# plutôt que de réécrire le data file entier.

# Passer la table en MoR pour voir la différence
spark.sql("""
    ALTER TABLE polaris.versions_demo.transactions_v2
    SET TBLPROPERTIES ('write.delete.mode' = 'merge-on-read')
""")

print('=' * 55)
print('DELETE V2 — transactions_v2 (MoR)')
print('=' * 55)

files_before_v2 = spark.sql('SELECT count(*) AS n FROM polaris.versions_demo.transactions_v2.files').collect()[0]['n']

t0 = time.time()
spark.sql(f"DELETE FROM polaris.versions_demo.transactions_v2 WHERE customer_id = '{TARGET}'")
elapsed_v2 = round(time.time() - t0, 3)

# En V2 MoR, content = 0 = data files, content = 1 = positional deletes, content = 2 = equality deletes
files_v2 = count_delete_files('versions_demo', 'transactions_v2')
remaining_v2 = spark.sql(f"SELECT count(*) AS n FROM polaris.versions_demo.transactions_v2 WHERE customer_id = '{TARGET}'").collect()[0]['n']

print(f'  Temps d\'exécution   : {elapsed_v2}s')
print(f'  Lignes restantes    : {remaining_v2} (attendu: 0)')
print(f'  Mécanisme           : Positional delete file — data file intact')
if files_v2:
    print('  Fichiers par type :')
    CONTENT_LABELS = {0: 'DATA', 1: 'POSITION_DELETES', 2: 'EQUALITY_DELETES'}
    for row in files_v2.collect():
        label = CONTENT_LABELS.get(row['content'], str(row['content']))
        print(f'     {label:<22} × {row["n"]:>2}  ({row["total_bytes"]/1024:.1f} Ko)')
print(f'  ✅ Le data file original n\'a PAS été réécrit — seulement un petit delete file ajouté')
print(f'  ⚠️  Après N deletes, la lecture devient lente — prévoir la compaction')

DELETE V2 — transactions_v2 (MoR)
  Temps d'exécution   : 2.033s
  Lignes restantes    : 0 (attendu: 0)
  Mécanisme           : Positional delete file — data file intact
  Fichiers par type :
     POSITION_DELETES       ×  2  (2.9 Ko)
     DATA                   ×  4  (10.4 Ko)
  ✅ Le data file original n'a PAS été réécrit — seulement un petit delete file ajouté
  ⚠️  Après N deletes, la lecture devient lente — prévoir la compaction


In [6]:
# ── DELETE sur V3 (Deletion Vectors) ─────────────────────────────────────────
# V3 introduit les Deletion Vectors : un format plus compact pour les deletes
# Stockés dans des fichiers Puffin (.puffin) plutôt que des positional delete files

print('=' * 55)
print('DELETE V3 — transactions_v3 (Deletion Vectors)')
print('=' * 55)

# Activer les Deletion Vectors (V3)
try:
    spark.sql("""
        ALTER TABLE polaris.versions_demo.transactions_v3
        SET TBLPROPERTIES (
            'write.delete.mode' = 'merge-on-read',
            'write.use-deletion-vectors' = 'true'
        )
    """)
    print('  Deletion Vectors activés (note : supporté nativement en Spark 4.0+ / Iceberg 1.6+)')
except Exception as e:
    print(f'  Note: Deletion Vectors config: {e}')
    print('  (Spark 3.5+ requis pour les DVs natifs — mode MoR utilisé)')

files_before_v3 = spark.sql('SELECT count(*) AS n FROM polaris.versions_demo.transactions_v3.files').collect()[0]['n']

t0 = time.time()
spark.sql(f"DELETE FROM polaris.versions_demo.transactions_v3 WHERE customer_id = '{TARGET}'")
elapsed_v3 = round(time.time() - t0, 3)

remaining_v3 = spark.sql(f"SELECT count(*) AS n FROM polaris.versions_demo.transactions_v3 WHERE customer_id = '{TARGET}'").collect()[0]['n']

print(f'  Temps d\'exécution   : {elapsed_v3}s')
print(f'  Lignes restantes    : {remaining_v3} (attendu: 0)')

# Observer les fichiers puffin (DVs) si Spark 3.5+
print('\n  Fichiers metadata après DELETE V3 :')
list_metadata_files('versions_demo', 'transactions_v3')

print(f'\n📊 Comparaison des mécanismes de DELETE :')
print(f'   V1  → {elapsed_v1}s  : CoW — réécriture complète du data file')
print(f'   V2  → {elapsed_v2}s  : MoR — positional delete file ajouté')
print(f'   V3  → {elapsed_v3}s  : MoR + DVs — bitmap compact (si supporté)')
print()
print('  En V3, les Deletion Vectors stockent un bitmap des positions supprimées')
print('  directement dans un fichier Puffin — jusqu\'à 10× plus compact que')
print('  les positional delete files V2 sur des tables à haute cardinalité.')

DELETE V3 — transactions_v3 (Deletion Vectors)
  Deletion Vectors activés (note : supporté nativement en Spark 4.0+ / Iceberg 1.6+)
  Temps d'exécution   : 0.924s
  Lignes restantes    : 0 (attendu: 0)

  Fichiers metadata après DELETE V3 :

📊 Comparaison des mécanismes de DELETE :
   V1  → 11.06s  : CoW — réécriture complète du data file
   V2  → 2.033s  : MoR — positional delete file ajouté
   V3  → 0.924s  : MoR + DVs — bitmap compact (si supporté)

  En V3, les Deletion Vectors stockent un bitmap des positions supprimées
  directement dans un fichier Puffin — jusqu'à 10× plus compact que
  les positional delete files V2 sur des tables à haute cardinalité.


### Partie B — Type Variant V3 : la killer feature pour le ML (10 min)

> Le type `Variant` permet de stocker des données semi-structurées (JSON-like)
> dans une colonne Iceberg sans schéma fixe — idéal pour les features ML
> dont le schéma évolue à chaque expérience.

In [7]:
# ── Type Variant (V3 uniquement) ──────────────────────────────────────────────
# Tenter de créer une table V3 avec une colonne Variant
# ⚠️  Type VARIANT (Iceberg V3) n'est pas encore supporté par Spark 3.5
# Spark 4.0+ requis pour la syntaxe native PARSE_JSON/features:field
# → On simule avec to_json() + get_json_object() (Spark 3.5 compatible)
# → En Spark 4.0+ ce serait : PARSE_JSON(json_str) et features:field_name

print('📋 Type Variant — colonnes semi-structurées sans schéma fixe\n')

try:
    spark.sql('DROP TABLE IF EXISTS polaris.versions_demo.features_variant_v3')
    spark.sql("""
        CREATE TABLE polaris.versions_demo.features_variant_v3 (
            customer_id  STRING,
            model_name   STRING,
            features     VARIANT,
            created_at   TIMESTAMP
        )
        USING iceberg
        TBLPROPERTIES ('format-version' = '3')
    """)

    # Insérer des features ML avec des schémas différents par client
    spark.sql("""
        INSERT INTO polaris.versions_demo.features_variant_v3 VALUES
        ('cust-001', 'churn-v1',
         to_json(struct(12 AS recency, 8 AS frequency, 45000 AS monetary, 'champion' AS segment)),
         current_timestamp()),
        ('cust-002', 'churn-v2',
         to_json(struct(45 AS recency, 2 AS frequency, 5000 AS monetary, 'at-risk' AS segment, 0.78 AS churn_score)),
         current_timestamp()),
        ('cust-003', 'recommend-v1',
         to_json(struct(array('Maison','Sport') AS preferred_categories, 80000 AS avg_basket, 'low' AS price_sensitivity)),
         current_timestamp())
    """)

    print('✅ Table avec colonne Variant créée (V3)')
    spark.sql("""
        SELECT customer_id, model_name,
               get_json_object(features,'$.segment') AS segment,
               get_json_object(features,'$.churn_score') AS churn_score,
               get_json_object(features,'$.recency') AS recency
        FROM polaris.versions_demo.features_variant_v3
    """).show(truncate=False)
    print('💡 Chaque client a un schéma différent dans la colonne Variant.')
    print('   En V1 ou V2, il faudrait un schéma fixe ou stocker du JSON en STRING (sans query pushdown).')

except Exception as e:
    print(f'⚠️  Type Variant non supporté par cette version de Spark/Iceberg: {e}')
    print()
    print('📋 Pourquoi Variant est important malgré le support limité :')
    print()
    print('  Situation actuelle sans Variant (V1/V2) :')
    print('  → Stocker les features ML en STRING(JSON) — aucun pushdown possible')
    print('  → Schéma fixe avec colonnes nullable — évolution difficile')
    print('  → Table MAP<STRING,DOUBLE> — perd le typage par feature')
    print()
    print('  Avec Variant (V3, Spark 3.5+) :')
    print('  → Colonnes semi-structurées avec pushdown : features:churn_score')
    print('  → Pas de migration de schéma quand le modèle ML change')
    print('  → Compatible avec le Feature Store : une colonne = toutes les features')

    # Simuler le comportement avec STRING (workaround V1/V2)
    spark.sql('DROP TABLE IF EXISTS polaris.versions_demo.features_json_v2')
    spark.sql("""
        CREATE TABLE polaris.versions_demo.features_json_v2 (
            customer_id STRING, model_name STRING,
            features_json STRING, created_at TIMESTAMP
        )
        USING iceberg
        TBLPROPERTIES ('format-version' = '2')
    """)
    spark.sql("""
        INSERT INTO polaris.versions_demo.features_json_v2 VALUES
        ('cust-001','churn-v1','{"recency":12,"frequency":8}',current_timestamp()),
        ('cust-002','churn-v2','{"recency":45,"churn_score":0.78}',current_timestamp())
    """)
    print()
    print('  Simulation V2 avec STRING(JSON) — le workaround actuel :')
    spark.sql("""
        SELECT customer_id,
               CAST(get_json_object(features_json, '$.recency') AS INT) AS recency,
               CAST(get_json_object(features_json, '$.churn_score') AS DOUBLE) AS churn_score
        FROM polaris.versions_demo.features_json_v2
    """).show()
    print('  ↑ get_json_object() = pas de pushdown → full scan à chaque requête')

📋 Type Variant — colonnes semi-structurées sans schéma fixe

⚠️  Type Variant non supporté par cette version de Spark/Iceberg: 
[UNSUPPORTED_DATATYPE] Unsupported data type "VARIANT".(line 5, pos 25)

== SQL ==

        CREATE TABLE polaris.versions_demo.features_variant_v3 (
            customer_id  STRING,
            model_name   STRING,
            features     VARIANT,
-------------------------^^^
            created_at   TIMESTAMP
        )
        USING iceberg
        TBLPROPERTIES ('format-version' = '3')
    


📋 Pourquoi Variant est important malgré le support limité :

  Situation actuelle sans Variant (V1/V2) :
  → Stocker les features ML en STRING(JSON) — aucun pushdown possible
  → Schéma fixe avec colonnes nullable — évolution difficile
  → Table MAP<STRING,DOUBLE> — perd le typage par feature

  Avec Variant (V3, Spark 3.5+) :
  → Colonnes semi-structurées avec pushdown : features:churn_score
  → Pas de migration de schéma quand le modèle ML change
  → Compatible avec 

---
## ⏱️ Temps 3 — Migration V1 → V2 → V3 (5 min)

> Démonstration du chemin de migration et du point de non-retour V3.

In [8]:
# ── Migration V1 → V2 → V3 : le chemin et le point de non-retour ─────────────

print('🔄 Démonstration du chemin de migration\n')

# Créer une table de test en V1
spark.sql('DROP TABLE IF EXISTS polaris.versions_demo.migration_test')
spark.sql("""
    CREATE TABLE polaris.versions_demo.migration_test
    (id STRING, val DOUBLE)
    USING iceberg
    TBLPROPERTIES ('format-version' = '1')
""")
spark.sql("INSERT INTO polaris.versions_demo.migration_test VALUES ('a', 1.0), ('b', 2.0)")

def get_format_version(table_full):
    try:
        rows = spark.sql(f"SELECT value FROM {table_full}.properties WHERE key = 'format-version'").collect()
        return rows[0]['value'] if rows else 'N/A'
    except Exception:
        return 'N/A'

T = 'polaris.versions_demo.migration_test'
print(f'  État initial        : format-version = {get_format_version(T)}')

# Migration V1 → V2 (non-destructive, pas de réécriture)
t0 = time.time()
spark.sql(f"ALTER TABLE {T} SET TBLPROPERTIES ('format-version' = '2')")
print(f'  Après migration V2  : format-version = {get_format_version(T)}  ({round(time.time()-t0,2)}s)')
print(f'  → Aucune réécriture des données existantes. Les fichiers V1 restent lisibles.')

# Vérifier qu'on peut faire un DELETE (impossible en V1)
spark.sql(f"DELETE FROM {T} WHERE id = 'a'")
remaining = spark.sql(f"SELECT count(*) AS n FROM {T}").collect()[0]['n']
print(f'  DELETE après migration V2 : {remaining} ligne(s) restante(s) ✅')

# Migration V2 → V3 (IRRÉVERSIBLE)
print()
print('  ⚠️  Migration V2 → V3 — IRRÉVERSIBLE')
try:
    t0 = time.time()
    spark.sql(f"ALTER TABLE {T} SET TBLPROPERTIES ('format-version' = '3')")
    print(f'  Après migration V3  : format-version = {get_format_version(T)}  ({round(time.time()-t0,2)}s)')

    # Tenter un retour en V2 — doit échouer
    print()
    print('  Tentative de retour en V2 (doit échouer) :')
    try:
        spark.sql(f"ALTER TABLE {T} SET TBLPROPERTIES ('format-version' = '2')")
        print('  ⚠️  Retour V2 accepté — comportement inattendu dans cette version')
    except Exception as e:
        print(f'  ✅ Erreur attendue : {str(e)[:80]}')
        print('  → Confirmed : V2 → V3 est irréversible dans le spec Iceberg')
except Exception as e:
    print(f'  V3 migration: {e}')

print()
print('📋 Règle de migration :')
print('   V1 → V2 : ✅ Non-destructif, pas de réécriture, engines V1 lisent encore les anciens fichiers')
print('   V2 → V3 : ⚠️  Irréversible. Trino et Flink (2024) ne liront plus les nouvelles données.')
print('   V3 → V2 : ❌ Impossible — jamais en production')

🔄 Démonstration du chemin de migration

  État initial        : format-version = N/A
  Après migration V2  : format-version = N/A  (0.26s)
  → Aucune réécriture des données existantes. Les fichiers V1 restent lisibles.
  DELETE après migration V2 : 1 ligne(s) restante(s) ✅

  ⚠️  Migration V2 → V3 — IRRÉVERSIBLE
  Après migration V3  : format-version = N/A  (0.31s)

  Tentative de retour en V2 (doit échouer) :
  ✅ Erreur attendue : An error occurred while calling o84.sql.
: org.apache.spark.SparkException: Unsu
  → Confirmed : V2 → V3 est irréversible dans le spec Iceberg

📋 Règle de migration :
   V1 → V2 : ✅ Non-destructif, pas de réécriture, engines V1 lisent encore les anciens fichiers
   V2 → V3 : ⚠️  Irréversible. Trino et Flink (2024) ne liront plus les nouvelles données.
   V3 → V2 : ❌ Impossible — jamais en production


---
## ⏱️ Extension — Benchmark DELETE × 100 (optionnel, 10 min)

In [9]:
# ── Benchmark : 100 DELETEs unitaires sur V1 vs V2 ───────────────────────────
# Simulate des suppressions RGPD haute fréquence

import numpy as np

print('⏳ Benchmark : 100 DELETEs unitaires (simulation RGPD haute fréquence)...')
print('   (Réduire à 10 si trop lent en Codespaces)\n')

N_BENCH = 20  # Réduire si nécessaire

# Recréer les tables de benchmark avec beaucoup de données
for version, mode in [(1, None), (2, 'merge-on-read')]:
    spark.sql(f'DROP TABLE IF EXISTS polaris.versions_demo.bench_v{version}')
    props = f"'format-version' = '{version}'"
    if mode:
        props += f", 'write.delete.mode' = '{mode}'"
    spark.sql(f"""
        CREATE TABLE polaris.versions_demo.bench_v{version}
        (id STRING, customer_id STRING, amount DOUBLE)
        USING iceberg TBLPROPERTIES ({props})
    """)
    # Insérer 1000 lignes avec des IDs uniques
    vals = ','.join([f"('id-{i}','cust-{i % 100:03d}',{round(50 + i*0.5, 2)})" for i in range(1000)])
    spark.sql(f'INSERT INTO polaris.versions_demo.bench_v{version} VALUES {vals}')

results = {}
for version in [1, 2]:
    times = []
    for i in range(N_BENCH):
        target_id = f'id-{i * 5}'  # IDs distincts
        t0 = time.time()
        spark.sql(f"DELETE FROM polaris.versions_demo.bench_v{version} WHERE id = '{target_id}'")
        times.append(time.time() - t0)
    results[version] = times
    avg = round(sum(times) / len(times), 3)
    print(f'  V{version} (×{N_BENCH} DELETEs) : moyenne {avg}s  min {round(min(times),3)}s  max {round(max(times),3)}s')

# Compter les delete files accumulés en V2
v2_files = count_delete_files('versions_demo', 'bench_v2')
if v2_files:
    print(f'\n  Fichiers accumulés après {N_BENCH} DELETEs en V2 MoR :')
    for row in v2_files.collect():
        LABELS = {0:'DATA', 1:'POSITION_DELETES', 2:'EQUALITY_DELETES'}
        print(f'     {LABELS.get(row["content"],"?"):<22} × {row["n"]:>3}  ({row["total_bytes"]/1024:.1f} Ko)')
    print(f'\n  → C\'est pourquoi la compaction est nécessaire en V2 MoR après N suppressions.')

speedup = round((sum(results[1])/len(results[1])) / max(sum(results[2])/len(results[2]), 0.001), 1)
comparatif = 'lent' if speedup > 1 else 'rapide'
print(f'\n📊 V1 est {speedup}× plus {comparatif} que V2 MoR pour les DELETEs unitaires.')
print('  V1 : réécriture du fichier entier à chaque DELETE')
print('  V2 MoR : ajout d\'un micro fichier de delete — rapide à l\'écriture, coût à la lecture')

⏳ Benchmark : 100 DELETEs unitaires (simulation RGPD haute fréquence)...
   (Réduire à 10 si trop lent en Codespaces)

  V1 (×20 DELETEs) : moyenne 1.379s  min 0.648s  max 5.204s
  V2 (×20 DELETEs) : moyenne 0.862s  min 0.57s  max 1.223s

  Fichiers accumulés après 20 DELETEs en V2 MoR :
     POSITION_DELETES       ×   1  (1.5 Ko)
     DATA                   ×   4  (6.7 Ko)

  → C'est pourquoi la compaction est nécessaire en V2 MoR après N suppressions.

📊 V1 est 1.6× plus lent que V2 MoR pour les DELETEs unitaires.
  V1 : réécriture du fichier entier à chaque DELETE
  V2 MoR : ajout d'un micro fichier de delete — rapide à l'écriture, coût à la lecture


---
## ⏱️ Débrief client (5 min)

### Questions — rôle Client :

1. **"Toutes mes tables sont en V1. Quand est-ce que je dois migrer en V2 ?"**
   → Dès qu'il y a une table avec des données personnelles soumises au RGPD, ou un pipeline CDC/MERGE.

2. **"Vous me dites que V3 ne marche pas avec Trino. Mais si dans 6 mois Trino le supporte, je fais quoi ?"**
   → Exactement le bon horizon — V3 + Trino est en cours de développement. Vérifier les release notes Trino avant de migrer.

3. **"La migration V1→V2 ne réécrit pas les données — mais est-ce que ça ne crée pas de l'incohérence ?"**
   → Non. Les métadonnées sont versionnées. Les anciens fichiers V1 restent lisibles. Les nouveaux fichiers V2 ajoutent les delete files. Iceberg gère les deux.

4. **"Vous avez dit que V3 est irréversible. Comment vous protégez un client d'une migration accidentelle ?"**
   → Gouvernance : RBAC Polaris — seul le DBA peut ALTER TABLE changer la format-version. + Processus de validation avant migration en prod.

---

## Récapitulatif

| Opération | V1 | V2 CoW | V2 MoR | V3 DVs |
|-----------|----|---------|---------|---------|
| DELETE unitaire | Lent (réécriture) | Lent (réécriture) | Rapide (delete file) | Très rapide (bitmap) |
| Lecture post-DELETE | Rapide | Rapide | Lente (merge) | Rapide (bitmap compact) |
| RGPD < 48h | ❌ Difficile | ✅ | ✅ | ✅ |
| MERGE INTO | ❌ | ✅ | ✅ | ✅ |
| Variant / JSON | ❌ | ❌ | ❌ | ✅ |
| Compatibilité Trino | ✅ | ✅ | ✅ | ❌ |
| Compatibilité Flink | ✅ | ✅ | ✅ | ❌ |

**Décision par défaut en production :**
- Tables append-only → **V1** (max compatibilité)
- Tout le reste → **V2** avec CoW ou MoR selon le workload
- Tables ML features avec Spark exclusif → **V3 Variant** (surveiller le support Trino)

In [10]:
# Nettoyage optionnel
print('Nettoyage :')
for t in ['transactions_v1','transactions_v2','transactions_v3',
          'features_variant_v3','features_json_v2',
          'migration_test','bench_v1','bench_v2']:
    try:
        spark.sql(f'DROP TABLE IF EXISTS polaris.versions_demo.{t}')
        print(f'  ✓ {t}')
    except: pass

Nettoyage :
  ✓ transactions_v1
  ✓ transactions_v2
  ✓ transactions_v3
  ✓ features_variant_v3
  ✓ features_json_v2
  ✓ migration_test
  ✓ bench_v1
  ✓ bench_v2
